# 03 — Supervisionado: pilotos na coorte definitiva

## Objetivo

Executar os pilotos supervisionados sobre o artefato central produzido a
partir da coorte definitiva, sem recriar features, alvo ou partições dentro
do notebook.

Este notebook:

- usa `configs/supervised.yaml` e o manifesto do artefato como contratos;
- lê somente `train_ref`, `val_tune`, `val_cal` e `val_meas`;
- mantém `test_ref` e `test_pandemic` fechados;
- preserva `id_registro` nas previsões e artefatos de auditoria;
- ajusta imputação, codificação e escala somente em `train_ref`;
- compara modelos apenas nos blocos internos de desenvolvimento;
- registra métricas, previsões e evidências sem promover pilotos a modelo final.

> **Limite desta execução:** não realiza abertura do teste, calibração final,
> escolha operacional de limiar nem interpretação clínica definitiva. As
> decisões posteriores devem respeitar os gates definidos no projeto.

## 1. Bootstrap, ambiente e imports

A célula localiza automaticamente a raiz do projeto. Para validação técnica isolada, o notebook também aceita `SRAG_NOTEBOOK_SMOKE=1`, usando dados sintéticos sem acessar o Parquet real.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display


def encontrar_raiz_projeto() -> Path:
    atual = Path.cwd().resolve()
    for pasta in [atual, *atual.parents]:
        if (pasta / "src").is_dir() and (pasta / "configs").is_dir():
            return pasta
    if os.getenv("SRAG_NOTEBOOK_SMOKE") == "1":
        return atual
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto.")


ROOT_DIR = encontrar_raiz_projeto()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

SMOKE_MODE = os.getenv("SRAG_NOTEBOOK_SMOKE") == "1"
STATUS_COORTE = "definitiva"
VERSAO_EXECUCAO = "artefato_central_v1"
SEED_FALLBACK = 42

print("ROOT_DIR:", ROOT_DIR)
print("SMOKE_MODE:", SMOKE_MODE)
print("Python:", sys.version.split()[0])
print("Sistema:", platform.system(), platform.release())

## 2. Imports do projeto e contrato ativo

No projeto real, esta célula reutiliza configuração, leakage, métricas, modelos e registro de execução existentes em `src/`. No modo de smoke test, somente as dependências públicas necessárias são carregadas.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.utils import config as C
from src.utils import env
from src.utils.reproducibility import Run
from src.evaluation.leakage import check_leakage
from src.evaluation.metrics import metricas_completas
from src.models.pipeline import criar_alvo_binario

env.carregar_dotenv()
env.garantir_diretorios()

CFG = C.carregar_supervised()

SEED = int(
    CFG.get(
        "temporal",
        {},
    ).get(
        "seed",
        C.SEED,
    )
)

TABLES_DIR = C.TABLES_DIR
PREDICTIONS_DIR = C.PREDICTIONS_DIR
RUNS_DIR = C.RUNS_DIR

for pasta in (
    TABLES_DIR,
    PREDICTIONS_DIR,
    RUNS_DIR,
):
    pasta.mkdir(
        parents=True,
        exist_ok=True,
    )

print(
    "Seed:",
    SEED,
)

print(
    "Modo de split:",
    CFG["temporal"]["modo"],
)

print(
    "Alvo bruto:",
    CFG["target"]["coluna"],
)

print(
    "Coluna temporal:",
    CFG["coluna_tempo"],
)

## 3. Consumir o artefato supervisionado definitivo

O notebook não reconstrói a coorte, as features, o alvo ou a partição-mestre.
Essas responsabilidades pertencem a `src.data.supervised_artifact`.

Nesta execução são carregados exclusivamente os quatro blocos de
desenvolvimento. `test_ref` e `test_pandemic` permanecem fechados.

In [ ]:
from src.data.supervised_artifact import load_supervised_artifact

BLOCOS_DESENVOLVIMENTO = (
    "train_ref",
    "val_tune",
    "val_cal",
    "val_meas",
)

df_supervisionado, manifest = load_supervised_artifact(
    root=ROOT_DIR,
    expected_kind="final",
    partitions=BLOCOS_DESENVOLVIMENTO,
)

COLUNA_PARTICAO = str(manifest["partition_column"])
ALVO_MODELO = str(manifest["target_column"])
COLUNA_ANO = "ano_internacao"
ALVO_BRUTO = str(CFG["target"]["coluna"])
COLUNA_TEMPO = str(CFG["coluna_tempo"])

FEATURES_PILOTO = list(manifest["feature_columns"])
FEATURES_NUMERICAS = list(manifest.get("numeric_columns") or [])
FEATURES_CATEGORICAS = list(manifest.get("categorical_columns") or [])
COLUNAS_AUDITORIA = list(manifest.get("audit_columns") or [])

TIPO_COORTE = str(manifest.get("cohort_kind"))
COHORT_KIND = TIPO_COORTE
FONTE_COORTE = dict(manifest.get("source") or {})
CAMINHO_COORTE = Path(
    str(FONTE_COORTE.get("path") or ROOT_DIR / "data" / "analytical" / "coorte_final.parquet")
)

ajustes_aplicados = [
    {
        "tipo": "artefato_central",
        "origem": "src.data.supervised_artifact",
        "destino": str(manifest.get("dataset_path")),
    }
]

if TIPO_COORTE != "final":
    raise RuntimeError("O B_07 exige o artefato produzido com a coorte final.")
if manifest.get("test_opened") is not False:
    raise RuntimeError("O manifesto indica abertura indevida do teste.")

print("Artefato supervisionado definitivo carregado.")
print("Linhas carregadas:", f"{len(df_supervisionado):,}")
print("Features:", len(FEATURES_PILOTO))
print("Blocos:", sorted(df_supervisionado[COLUNA_PARTICAO].astype("string").unique()))
print("Teste aberto:", manifest.get("test_opened"))

## 4. Validar o contrato do artefato

As verificações desta seção confirmam o esquema mínimo necessário para os
pilotos. Nenhuma feature é criada, removida ou selecionada aqui.

In [ ]:
colunas_obrigatorias = {
    *FEATURES_PILOTO,
    "id_registro",
    "group_id",
    COLUNA_ANO,
    ALVO_MODELO,
    COLUNA_PARTICAO,
}
colunas_ausentes = sorted(colunas_obrigatorias - set(df_supervisionado.columns))
blocos_observados = set(
    df_supervisionado[COLUNA_PARTICAO].astype("string").unique()
)
alvo_observado = set(
    pd.to_numeric(df_supervisionado[ALVO_MODELO], errors="coerce")
    .dropna().astype(int).unique()
)
grupos_por_bloco = df_supervisionado.groupby(
    "group_id", observed=True
)[COLUNA_PARTICAO].nunique()

contrato = pd.DataFrame(
    {
        "verificacao": [
            "artefato_nao_vazio",
            "coorte_final",
            "teste_fechado",
            "colunas_obrigatorias",
            "alvo_binario",
            "blocos_de_desenvolvimento",
            "id_registro_unico",
            "grupos_disjuntos",
        ],
        "resultado": [
            not df_supervisionado.empty,
            TIPO_COORTE == "final",
            manifest.get("test_opened") is False,
            not colunas_ausentes,
            alvo_observado == {0, 1},
            blocos_observados == set(BLOCOS_DESENVOLVIMENTO),
            not df_supervisionado["id_registro"].duplicated().any(),
            int((grupos_por_bloco > 1).sum()) == 0,
        ],
    }
)

display(contrato)
if not bool(contrato["resultado"].all()):
    falhas = contrato.loc[~contrato["resultado"], "verificacao"].tolist()
    raise RuntimeError(f"Contrato do artefato falhou em: {falhas}")

print("Contrato do artefato validado.")

## 5. Preparar treino e validações internas

`train_ref` é usado para ajuste. `val_tune`, `val_cal` e `val_meas` são
mantidos identificados no DataFrame de validação. Nenhuma partição de teste
é carregada.

In [ ]:
treino = df_supervisionado.loc[
    df_supervisionado[COLUNA_PARTICAO].eq("train_ref")
].copy()
validacao = df_supervisionado.loc[
    df_supervisionado[COLUNA_PARTICAO].isin(("val_tune", "val_cal", "val_meas"))
].copy()

X_treino = treino.loc[:, FEATURES_PILOTO]
y_treino = pd.to_numeric(treino[ALVO_MODELO], errors="raise").astype("int8")
X_validacao = validacao.loc[:, FEATURES_PILOTO]
y_validacao = pd.to_numeric(validacao[ALVO_MODELO], errors="raise").astype("int8")

blocos_treino = sorted(treino[COLUNA_PARTICAO].astype("string").unique())
blocos_validacao = sorted(validacao[COLUNA_PARTICAO].astype("string").unique())
sobreposicao_grupos = len(set(treino["group_id"]) & set(validacao["group_id"]))

if blocos_treino != ["train_ref"]:
    raise RuntimeError(f"Bloco de treino inesperado: {blocos_treino}")
if set(blocos_validacao) != {"val_tune", "val_cal", "val_meas"}:
    raise RuntimeError(f"Blocos de validação inesperados: {blocos_validacao}")
if sobreposicao_grupos:
    raise RuntimeError("Há group_id compartilhado entre treino e validação.")

resumo_base = pd.DataFrame(
    {
        "metrica": [
            "status_coorte", "versao_execucao", "linhas_artefato_carregadas",
            "linhas_treino", "linhas_validacao", "positivos_treino",
            "positivos_validacao", "teste_aberto",
        ],
        "valor": [
            STATUS_COORTE, VERSAO_EXECUCAO, len(df_supervisionado), len(treino),
            len(validacao), int(y_treino.sum()), int(y_validacao.sum()),
            manifest.get("test_opened"),
        ],
    }
)

ausencia_features = pd.DataFrame(
    {
        "feature": FEATURES_PILOTO,
        "tipo_configurado": [
            "numerica" if coluna in FEATURES_NUMERICAS else "categorica"
            for coluna in FEATURES_PILOTO
        ],
        "dtype": [str(df_supervisionado[coluna].dtype) for coluna in FEATURES_PILOTO],
        "ausentes": [
            int(df_supervisionado[coluna].isna().sum())
            for coluna in FEATURES_PILOTO
        ],
    }
)
ausencia_features["percentual_ausente"] = (
    ausencia_features["ausentes"] / len(df_supervisionado) * 100.0
)

display(resumo_base)
display(ausencia_features.sort_values("percentual_ausente", ascending=False))
print("Treino:", X_treino.shape, "positivos:", int(y_treino.sum()))
print("Validação:", X_validacao.shape, "positivos:", int(y_validacao.sum()))
print("Blocos em treino:", blocos_treino)
print("Blocos em validação:", blocos_validacao)

In [ ]:
# ============================================================
# 5.1 Teste do pré-processador central esparso
# ============================================================

import gc

import numpy as np
from scipy import sparse
from typing import cast

from src.models.pipeline import (
    FEATURES_CATEGORICAS as FEATURES_CATEGORICAS_SRC,
    FEATURES_NUMERICAS as FEATURES_NUMERICAS_SRC,
    build_preprocessor,
)

features_numericas_notebook = list(FEATURES_NUMERICAS)
features_categoricas_notebook = list(FEATURES_CATEGORICAS)
features_numericas_src = list(FEATURES_NUMERICAS_SRC)
features_categoricas_src = list(FEATURES_CATEGORICAS_SRC)

if features_numericas_notebook != features_numericas_src:
    raise RuntimeError(
        "As features numéricas do notebook diferem das features "
        f"numéricas de src.models.pipeline. Notebook: {features_numericas_notebook}. "
        f"src: {features_numericas_src}"
    )

if features_categoricas_notebook != features_categoricas_src:
    raise RuntimeError(
        "As features categóricas do notebook diferem das features "
        f"categóricas de src.models.pipeline. Notebook: {features_categoricas_notebook}. "
        f"src: {features_categoricas_src}"
    )

features_teste_real = features_numericas_src + features_categoricas_src

colunas_ausentes_teste_real = [
    coluna
    for coluna in features_teste_real
    if coluna not in X_treino.columns
]

if colunas_ausentes_teste_real:
    raise KeyError(
        "X_treino não contém todas as features esperadas: "
        f"{colunas_ausentes_teste_real}"
    )

QUANTIDADE_MAXIMA_TESTE_REAL = 50_000
quantidade_linhas_teste_real = min(
    QUANTIDADE_MAXIMA_TESTE_REAL,
    len(X_treino),
)

X_teste_real_preprocessador = X_treino.loc[
    :, features_teste_real
].head(quantidade_linhas_teste_real)

preprocessador_teste_real = build_preprocessor(
    numericas=features_numericas_src,
    categoricas=features_categoricas_src,
    saida="esparsa",
)

matriz_teste_real = preprocessador_teste_real.fit_transform(
    X_teste_real_preprocessador
)

resultado_esparso_teste_real = sparse.issparse(matriz_teste_real)
dtype_teste_real = np.dtype(cast(sparse.csr_matrix, matriz_teste_real).dtype)

if not resultado_esparso_teste_real:
    raise AssertionError(
        "O preprocessor central produziu uma matriz densa. "
        f"Tipo encontrado: {type(matriz_teste_real).__name__}"
    )

if dtype_teste_real != np.dtype("float32"):
    raise AssertionError(
        "O preprocessor central deveria produzir float32, "
        f"mas produziu {dtype_teste_real}."
    )

if matriz_teste_real.shape[0] != quantidade_linhas_teste_real:
    raise AssertionError(
        "A quantidade de linhas mudou durante o processamento."
    )

quantidade_celulas_teste_real = np.prod(
    matriz_teste_real.shape,
    dtype=np.int64,
)

quantidade_nao_nulos_teste_real = int(
    np.count_nonzero(cast(sparse.csr_matrix, matriz_teste_real).data)
)
densidade_teste_real = (
    quantidade_nao_nulos_teste_real / quantidade_celulas_teste_real
)

matriz_teste_real_csr = sparse.csr_matrix(matriz_teste_real)

memoria_esparsa_teste_real_mb = (
    matriz_teste_real_csr.data.nbytes
    + matriz_teste_real_csr.indices.nbytes
    + matriz_teste_real_csr.indptr.nbytes
) / 1_048_576

memoria_densa_teste_real_estimada_mb = (
    quantidade_celulas_teste_real
    / 1_048_576
    / (1 / dtype_teste_real.itemsize)
)

economia_estimada_percentual = 100 - (
    memoria_esparsa_teste_real_mb
    / memoria_densa_teste_real_estimada_mb
    / 0.01
)

print("Contrato central confirmado.")
print("Features numéricas:", len(features_numericas_src))
print("Features categóricas:", len(features_categoricas_src))
print("Total de features originais:", len(features_teste_real))
print("Dimensão da entrada:", X_teste_real_preprocessador.shape)
print("Tipo da saída:", type(matriz_teste_real).__name__)
print("Dimensão da saída:", matriz_teste_real.shape)
print("É esparsa:", resultado_esparso_teste_real)
print("Dtype:", dtype_teste_real)
print("Valores não nulos:", f"{quantidade_nao_nulos_teste_real:,}")
print("Densidade:", f"{densidade_teste_real:.2%}")
print(
    "Memória esparsa observada:",
    f"{memoria_esparsa_teste_real_mb:,.2f} MB",
)
print(
    "Memória densa estimada em float32:",
    f"{memoria_densa_teste_real_estimada_mb:,.2f} MB",
)
print("Economia estimada:", f"{economia_estimada_percentual:,.2f}%")

del matriz_teste_real
del preprocessador_teste_real
del X_teste_real_preprocessador

gc.collect()

print("Teste real do preprocessor concluído com sucesso.")
print("X_treino foi preservado:", "X_treino" in globals())


In [ ]:
# ============================================================
# 5.2 Teste estrutural do boosting central
# ============================================================

import gc

import numpy as np

from src.models.boosting import build_boosting_pipeline
from src.models.pipeline import (
    FEATURES_CATEGORICAS as FEATURES_CATEGORICAS_BOOSTING,
    FEATURES_NUMERICAS as FEATURES_NUMERICAS_BOOSTING,
)

features_boosting_teste = (
    list(FEATURES_NUMERICAS_BOOSTING)
    + list(FEATURES_CATEGORICAS_BOOSTING)
)

colunas_ausentes_boosting = [
    coluna
    for coluna in features_boosting_teste
    if coluna not in X_treino.columns
]

if colunas_ausentes_boosting:
    raise KeyError(
        "X_treino não contém todas as features esperadas "
        f"pelo boosting central: {colunas_ausentes_boosting}"
    )

QUANTIDADE_LINHAS_BOOSTING_TESTE = min(
    10_000,
    len(X_treino),
)

X_boosting_teste = X_treino.loc[
    :, features_boosting_teste
].head(QUANTIDADE_LINHAS_BOOSTING_TESTE)

y_boosting_teste = y_treino.loc[X_boosting_teste.index]

if len(X_boosting_teste) != len(y_boosting_teste):
    raise AssertionError(
        "X e y ficaram desalinhados no teste do boosting."
    )

pipeline_boosting_teste = build_boosting_pipeline(
    max_iter=5,
)

preprocessador_boosting_teste = (
    pipeline_boosting_teste.named_steps["pre"]
)

classificador_boosting_teste = (
    pipeline_boosting_teste.named_steps["clf"]
)

matriz_ordinal_teste = preprocessador_boosting_teste.fit_transform(
    X_boosting_teste
)

if matriz_ordinal_teste.shape[0] != len(X_boosting_teste):
    raise AssertionError(
        "A quantidade de linhas mudou no processamento ordinal."
    )

if matriz_ordinal_teste.shape[1] != len(features_boosting_teste):
    raise AssertionError(
        "O preprocessor ordinal deveria produzir uma coluna "
        "por feature original."
    )

if np.dtype(matriz_ordinal_teste.dtype) != np.dtype("float32"):
    raise AssertionError(
        "O preprocessor ordinal deveria produzir float32, "
        f"mas produziu {matriz_ordinal_teste.dtype}."
    )

pipeline_boosting_teste.fit(
    X_boosting_teste,
    y_boosting_teste,
)

QUANTIDADE_PREDICOES_TESTE = min(
    1_000,
    len(X_boosting_teste),
)

probabilidades_boosting_teste = pipeline_boosting_teste.predict_proba(
    X_boosting_teste.head(QUANTIDADE_PREDICOES_TESTE)
)

formato_esperado_probabilidades = (
    QUANTIDADE_PREDICOES_TESTE,
    2,
)

if probabilidades_boosting_teste.shape != formato_esperado_probabilidades:
    raise AssertionError(
        "Formato inesperado das probabilidades. "
        f"Esperado: {formato_esperado_probabilidades}; "
        f"observado: {probabilidades_boosting_teste.shape}."
    )

if not np.isfinite(probabilidades_boosting_teste).all():
    raise AssertionError(
        "O boosting produziu probabilidades não finitas."
    )

probabilidades_validas = np.logical_and(
    probabilidades_boosting_teste >= 0,
    probabilidades_boosting_teste <= 1,
).all()

if not probabilidades_validas:
    raise AssertionError(
        "O boosting produziu probabilidades fora do intervalo válido."
    )

print("Etapas do pipeline:", list(pipeline_boosting_teste.named_steps.keys()))
print("Classificador:", type(classificador_boosting_teste).__name__)
print("Tipo da matriz ordinal:", type(matriz_ordinal_teste).__name__)
print("Dimensão da matriz ordinal:", matriz_ordinal_teste.shape)
print("Dtype da matriz ordinal:", matriz_ordinal_teste.dtype)
print("Formato das probabilidades:", probabilidades_boosting_teste.shape)
print("Teste estrutural do boosting concluído com sucesso.")

del matriz_ordinal_teste
del probabilidades_boosting_teste
del preprocessador_boosting_teste
del classificador_boosting_teste
del pipeline_boosting_teste
del X_boosting_teste
del y_boosting_teste

gc.collect()

print("Objetos temporários do teste do boosting removidos.")


## 6. Construir o pré-processamento

Numéricas: mediana + padronização. Categóricas: categoria explícita para ausência + one-hot que ignora categorias novas. O `fit` será executado exclusivamente no treino.

In [ ]:
from src.models.pipeline import build_preprocessor

preprocessador = build_preprocessor(
    numericas=list(FEATURES_NUMERICAS),
    categoricas=list(FEATURES_CATEGORICAS),
    saida="esparsa",
)

print(preprocessador)

## 7. Funções locais de avaliação do piloto

Estas funções apenas organizam resultados na validação. Não escolhem limiar e não acessam o teste.

In [ ]:
import gc
from sklearn.model_selection import train_test_split
from typing import Any, cast

for _nome_objeto in ["df", "df_supervisionado"]:
    if _nome_objeto in globals():
        del globals()[_nome_objeto]

gc.collect()

MAX_TREINO_ETAPA_INICIAL = 1_000_000


def amostrar_treino_estratificado(
    maximo: int,
    seed: int,
) -> tuple[pd.DataFrame, pd.Series]:
    if len(X_treino) <= maximo:
        return X_treino, y_treino

    estratos = (
        treino.loc[X_treino.index, COLUNA_ANO].astype(str)
        + "__y"
        + y_treino.astype(int).astype(str)
    )

    indices_amostra, _ = train_test_split(
        X_treino.index.to_numpy(),
        train_size=maximo,
        stratify=estratos.to_numpy(),
        random_state=seed,
    )

    indices_amostra = np.sort(indices_amostra)
    return (
        X_treino.loc[indices_amostra],
        y_treino.loc[indices_amostra],
    )


def executar_piloto(
    nome: str,
    pipeline: Pipeline,
    max_treino: int | None = None,
) -> tuple[Pipeline, dict, pd.DataFrame]:
    if max_treino is None:
        X_fit, y_fit = X_treino, y_treino
    else:
        X_fit, y_fit = amostrar_treino_estratificado(
            max_treino,
            SEED,
        )

    inicio = time.perf_counter()
    pipeline.fit(X_fit, y_fit)
    duracao_fit = time.perf_counter() - inicio

    inicio_pred = time.perf_counter()
    prob = pipeline.predict_proba(X_validacao)[:, 1]
    duracao_pred = time.perf_counter() - inicio_pred

    if not np.isfinite(prob).all():
        raise RuntimeError(
            f"{nome}: probabilidades não finitas."
        )

    if not ((prob >= 0) & (prob <= 1)).all():
        raise RuntimeError(
            f"{nome}: probabilidades fora de [0, 1]."
        )

    metricas = metricas_completas(
        y_validacao,
        prob,
        limiar=0.5,
    )

    if "auc" not in metricas:
        raise KeyError(
            "A função central de métricas não retornou 'auc'. "
            f"Chaves disponíveis: {sorted(metricas)}"
        )

    metricas["roc_auc"] = metricas["auc"]
    metricas["limiar_apenas_diagnostico"] = 0.5
    metricas["n_validacao"] = int(len(y_validacao))
    cast(dict[str, Any], metricas).update(
        {
            "modelo": nome,
            "status_coorte": STATUS_COORTE,
            "versao_execucao": VERSAO_EXECUCAO,
            "n_treino_usado": int(len(X_fit)),
            "duracao_fit_s": float(duracao_fit),
            "duracao_pred_s": float(duracao_pred),
        }
    )

    predicoes = pd.DataFrame(
        {
            "id_registro": validacao.loc[X_validacao.index, "id_registro"].to_numpy(),
            "y_true": y_validacao.to_numpy(),
            "y_prob": prob,
            "modelo": nome,
            "status_coorte": STATUS_COORTE,
        }
    )

    del X_fit, y_fit
    gc.collect()

    return pipeline, metricas, predicoes


print("Funções de avaliação carregadas.")
print(
    "Teto de treino da etapa inicial:",
    MAX_TREINO_ETAPA_INICIAL,
)

## 8. Executar o baseline Dummy

O Dummy define o piso mínimo. O modelo não será usado para decisão final.

In [ ]:
from src.models.baseline import build_baseline_pipeline

estrategia_dummy = (
    CFG.get("models", {})
    .get("baseline", {})
    .get("strategy", "prior")
)

pipeline_dummy = build_baseline_pipeline(
    strategy=estrategia_dummy,
    preprocessor="passthrough",
)

pipeline_dummy, metricas_dummy, predicoes_dummy = executar_piloto(
    "dummy_prior",
    pipeline_dummy,
)

display(pd.DataFrame([metricas_dummy]))

## 9. Executar a regressão logística piloto

Usamos `C=1.0` apenas como configuração neutra de integração. Esse valor não é tuning e não será tratado como hiperparâmetro final.

In [ ]:
from src.models.logistic import build_logistic_pipeline
from typing import Literal

log_cfg = CFG.get("models", {}).get("logistic", {})

solvers_validos = (
    "lbfgs",
    "liblinear",
    "newton-cg",
    "newton-cholesky",
    "sag",
    "saga",
)
solver_config = log_cfg.get("solver") or "lbfgs"

if solver_config not in solvers_validos:
    raise ValueError(f"Solver inválido: {solver_config!r}")

pipeline_logistica = build_logistic_pipeline(
    C=float(log_cfg.get("C") or 1.0),
    penalty=log_cfg.get("penalty"),
    solver=cast(
        Literal["lbfgs", "liblinear", "newton-cg", "newton-cholesky", "sag", "saga"],
        solver_config,
    ),
    max_iter=int(log_cfg.get("max_iter", 1000)),
    class_weight=log_cfg.get("class_weight", "balanced"),
    preprocessor=preprocessador,
)

pipeline_logistica, metricas_logistica, predicoes_logistica = executar_piloto(
    "logistica_piloto_C1",
    pipeline_logistica,
    max_treino=MAX_TREINO_ETAPA_INICIAL,
)

display(pd.DataFrame([metricas_logistica]))

## 10. Comparação de desenvolvimento na validação

A comparação confirma apenas que o pipeline aprende algo além do piso. Não seleciona finalista e não autoriza tuning.

In [ ]:
resultados_piloto = pd.DataFrame([
    metricas_dummy,
    metricas_logistica,
]).sort_values(
    "pr_auc",
    ascending=False,
).reset_index(drop=True)

colunas_exibicao = [
    "modelo",
    "pr_auc",
    "roc_auc",
    "brier",
    "recall",
    "precision",
    "f1",
    "accuracy",
    "duracao_fit_s",
    "duracao_pred_s",
    "status_coorte",
]

display(resultados_piloto[colunas_exibicao])

status_observados = set(
    resultados_piloto["status_coorte"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

status_esperado = {STATUS_COORTE}

if status_observados != status_esperado:
    raise RuntimeError(
        "Os resultados não possuem a marcação esperada da coorte. "
        f"Esperado: {sorted(status_esperado)}. "
        f"Observado: {sorted(status_observados)}."
    )

print("Status da coorte validado:", STATUS_COORTE)

## 11. Validações pós-treino

Esta célula confirma que o pré-processador foi ajustado, que as categorias pertencem ao treino e que as probabilidades de validação são válidas.

In [ ]:
pre_log = pipeline_logistica.named_steps["pre"]
nomes_features_transformadas = pre_log.get_feature_names_out()

validacoes_pos_treino = {
    "dummy_tem_predict_proba": hasattr(pipeline_dummy, "predict_proba"),
    "logistica_tem_predict_proba": hasattr(pipeline_logistica, "predict_proba"),
    "features_transformadas_nao_vazias": len(nomes_features_transformadas) > 0,
    "prob_dummy_validas": predicoes_dummy["y_prob"].between(0, 1).all(),
    "prob_logistica_validas": predicoes_logistica["y_prob"].between(0, 1).all(),
    "teste_nao_materializado": "teste" not in globals(),
    "sem_feature_proibida": not bool(set(FEATURES_PILOTO) & PROIBIDAS),
}

display(pd.DataFrame({
    "verificacao": validacoes_pos_treino.keys(),
    "resultado": validacoes_pos_treino.values(),
}))

if not all(validacoes_pos_treino.values()):
    falhas = [k for k, v in validacoes_pos_treino.items() if not v]
    raise RuntimeError(f"Falhas pós-treino: {falhas}")

print("Quantidade de features após transformação:", len(nomes_features_transformadas))
print("Primeiras features:", nomes_features_transformadas[:15].tolist())

## 12. Persistir artefatos de desenvolvimento

Não salvamos o modelo treinado como candidato final. Salvamos contrato, partições, métricas e predições de validação para auditoria.

In [ ]:
import json
from pathlib import Path

timestamp_execucao = (
    pd.Timestamp.now(tz="UTC")
    .strftime("%Y%m%dT%H%M%SZ")
)

prefixo = (
    f"03_pilotos_{STATUS_COORTE}_"
    f"{timestamp_execucao}"
)

caminho_contrato = (
    TABLES_DIR / f"{prefixo}_contrato.csv"
)
caminho_particoes = (
    TABLES_DIR / f"{prefixo}_particoes.csv"
)
caminho_metricas = (
    TABLES_DIR / f"{prefixo}_metricas_validacao.csv"
)
caminho_predicoes = (
    PREDICTIONS_DIR
    / f"{prefixo}_predicoes_validacao.parquet"
)
caminho_manifesto = (
    TABLES_DIR / f"{prefixo}_manifesto.json"
)

caminho_manifesto_mestre = (
    ROOT_DIR
    / "data"
    / "analytical"
    / "manifesto_particao.json"
)

caminho_predicoes_val_tune = (
    ROOT_DIR
    / "outputs"
    / "final"
    / "predicoes_val_tune_piloto.parquet"
)

caminho_predicoes_val_tune.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if not caminho_manifesto_mestre.is_file():
    raise FileNotFoundError(
        "Manifesto central da partição-mestre ausente: "
        f"{caminho_manifesto_mestre}"
    )

manifesto_mestre = json.loads(
    caminho_manifesto_mestre.read_text(
        encoding="utf-8"
    )
)

resumo_particoes = (
    df_supervisionado.groupby(
        COLUNA_PARTICAO,
        observed=True,
    )
    .agg(
        quantidade=("id_registro", "size"),
        grupos=("group_id", "nunique"),
        positivos=(ALVO_MODELO, "sum"),
    )
    .reset_index()
)

resumo_particoes["prevalencia"] = (
    resumo_particoes["positivos"]
    / resumo_particoes["quantidade"]
)

predicoes_validacao = pd.concat(
    [
        predicoes_dummy,
        predicoes_logistica,
    ],
    ignore_index=True,
)

if "id_registro" not in predicoes_validacao.columns:
    raise KeyError(
        "As previsões precisam conter id_registro."
    )

identificadores_validacao = (
    validacao.loc[
        :,
        [
            "id_registro",
            COLUNA_PARTICAO,
            COLUNA_ANO,
            "regiao_residencia",
            "sg_uf",
            "faixa_etaria",
            "sexo_categoria",
            ALVO_MODELO,
        ],
    ]
    .drop_duplicates(
        subset=["id_registro"],
        keep="first",
    )
)

predicoes_validacao = predicoes_validacao.merge(
    identificadores_validacao,
    on="id_registro",
    how="left",
    validate="many_to_one",
)

if predicoes_validacao[
    COLUNA_PARTICAO
].isna().any():
    raise RuntimeError(
        "Há previsões sem bloco após a junção "
        "por id_registro."
    )

contrato.to_csv(
    caminho_contrato,
    index=False,
    encoding="utf-8-sig",
)

resumo_particoes.to_csv(
    caminho_particoes,
    index=False,
    encoding="utf-8-sig",
)

resultados_piloto.to_csv(
    caminho_metricas,
    index=False,
    encoding="utf-8-sig",
)

predicoes_validacao.to_parquet(
    caminho_predicoes,
    index=False,
)

predicoes_val_tune = (
    predicoes_validacao.loc[
        predicoes_validacao[
            COLUNA_PARTICAO
        ].eq("val_tune")
    ]
    .copy()
)

if predicoes_val_tune.empty:
    raise RuntimeError(
        "Nenhuma previsão de val_tune foi produzida."
    )

if set(
    predicoes_val_tune[COLUNA_PARTICAO]
    .astype("string")
    .unique()
) != {"val_tune"}:
    raise RuntimeError(
        "O arquivo de val_tune contém outros blocos."
    )

predicoes_val_tune.to_parquet(
    caminho_predicoes_val_tune,
    index=False,
)

contagens_por_bloco = {
    str(row[COLUNA_PARTICAO]): int(
        row["quantidade"]
    )
    for _, row in resumo_particoes.iterrows()
}

manifesto_execucao = {
    "status_coorte": STATUS_COORTE,
    "cohort_kind": manifest.get(
        "cohort_kind"
    ),
    "versao_execucao": VERSAO_EXECUCAO,
    "fonte": manifest.get("source"),
    "dataset_path": manifest.get(
        "dataset_path"
    ),
    "dataset_sha256": manifest.get(
        "dataset_sha256"
    ),
    "seed": SEED,
    "features": FEATURES_PILOTO,
    "features_numericas": (
        FEATURES_NUMERICAS
    ),
    "features_categoricas": (
        FEATURES_CATEGORICAS
    ),
    "n_artefato_carregado": int(
        len(df_supervisionado)
    ),
    "n_treino": int(len(treino)),
    "n_validacao": int(len(validacao)),
    "contagens_por_bloco": (
        contagens_por_bloco
    ),
    "particao_mestra": {
        "manifesto": str(
            caminho_manifesto_mestre
        ),
        "metodo": manifesto_mestre.get(
            "metodo"
        ),
        "linhas_quarentena": (
            manifesto_mestre.get(
                "rows_quarantined"
            )
        ),
        "grupos_quarentena": (
            manifesto_mestre.get(
                "groups_quarantined"
            )
        ),
    },
    "teste_aberto": False,
    "blocos_teste_carregados": [],
    "artefatos": {
        "contrato": str(
            caminho_contrato
        ),
        "particoes": str(
            caminho_particoes
        ),
        "metricas_validacao": str(
            caminho_metricas
        ),
        "predicoes_validacao": str(
            caminho_predicoes
        ),
        "predicoes_val_tune": str(
            caminho_predicoes_val_tune
        ),
        "manifesto_particao_mestre": str(
            caminho_manifesto_mestre
        ),
    },
}

caminho_manifesto.write_text(
    json.dumps(
        manifesto_execucao,
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
    encoding="utf-8",
)

print("Artefatos definitivos gravados:")

for caminho in (
    caminho_contrato,
    caminho_particoes,
    caminho_metricas,
    caminho_predicoes,
    caminho_predicoes_val_tune,
    caminho_manifesto,
    caminho_manifesto_mestre,
):
    print("-", caminho)


## 13. Reabrir e validar os artefatos

A persistência somente é considerada concluída depois do round-trip.

In [ ]:
metricas_reabertas = pd.read_csv(caminho_metricas, encoding="utf-8-sig")
predicoes_reabertas = pd.read_parquet(caminho_predicoes)
manifesto_reaberto = json.loads(caminho_manifesto.read_text(encoding="utf-8"))
manifesto_mestre_reaberto = json.loads(
    caminho_manifesto_mestre.read_text(encoding="utf-8")
)

validacoes_artefatos = {
    "contrato_existe": caminho_contrato.exists(),
    "particoes_existe": caminho_particoes.exists(),
    "metricas_existe": caminho_metricas.exists(),
    "predicoes_existe": caminho_predicoes.exists(),
    "manifesto_existe": caminho_manifesto.exists(),
    "manifesto_mestre_existe": caminho_manifesto_mestre.exists(),
    "metricas_mesmas_linhas": len(metricas_reabertas) == len(resultados_piloto),
    "predicoes_esperadas": len(predicoes_reabertas) == 2 * len(validacao),
    "status_coorte_manifesto_correto": (
        manifesto_reaberto["status_coorte"] == STATUS_COORTE
    ),
    "blocos_mestre_reabertos": (
        set(manifesto_mestre_reaberto["blocos"])
        == set(MAPA_BLOCO_PARA_PARTICAO) | {"exploracao"}
    ),
    "teste_declarado_fechado": (
        manifesto_reaberto["teste_aberto"] is False
        and manifesto_reaberto["particao_mestra"]["test_ref_fechado"]
        and manifesto_reaberto["particao_mestra"]["test_pandemic_fechado"]
    ),
}

display(pd.DataFrame({
    "verificacao": validacoes_artefatos.keys(),
    "resultado": validacoes_artefatos.values(),
}))
if not all(validacoes_artefatos.values()):
    falhas = [k for k, v in validacoes_artefatos.items() if not v]
    raise RuntimeError(f"Falhas no round-trip dos artefatos: {falhas}")

## 14. Registrar a execução

No modo real, o notebook registra uma execução com `Run`. No smoke test isolado, cria um registro local simples apenas para validar o fluxo.

In [ ]:
with Run(
    modelo="03_supervisionado_pilotos",
    seed=SEED,
    parametros={
        "status_coorte": STATUS_COORTE,
        "versao_execucao": VERSAO_EXECUCAO,
        "split": manifesto_mestre["metodo"],
        "split_estratificacao": manifesto_mestre["estratificacao"],
        "teste_aberto": False,
    },
    dataset={
        "n_rows": int(len(base_modelagem)),
        "train_rows": int(len(treino)),
        "valid_rows": int(len(validacao)),
        "test_rows": int(
            manifesto_mestre["blocos"]["test_ref"]["n_linhas"]
            + manifesto_mestre["blocos"]["test_pandemic"]["n_linhas"]
        ),
    },
    features={
        "n_features": len(FEATURES_PILOTO),
        "usadas": FEATURES_PILOTO,
        "features_final_congelada": bool(FEATURES_FINAIS_CFG),
    },
    notas=(
        "Piloto desenvolvimento. "
        "Particao-mestre com seis blocos. "
        "Holdout (test_ref, test_pandemic) nao consultado. "
        "Sem tuning, calibracao ou selecao de limiar."
    ),
    dataset_path=CAMINHO_COORTE,
) as run:
    run.metrica(
        dummy_pr_auc=float(metricas_dummy["pr_auc"]),
        logistica_pr_auc=float(metricas_logistica["pr_auc"]),
        logistica_roc_auc=float(metricas_logistica["roc_auc"]),
        logistica_brier=float(metricas_logistica["brier"]),
    )

print("Run registrado em:", run.pasta)

## 15. Checklist de encerramento

O notebook está pronto para o próximo incremento somente se todos os itens abaixo forem verdadeiros.

In [ ]:
metricas_reabertas = pd.read_csv(caminho_metricas, encoding="utf-8-sig")
predicoes_reabertas = pd.read_parquet(caminho_predicoes)
manifesto_reaberto = json.loads(
    caminho_manifesto.read_text(encoding="utf-8")
)
manifesto_mestre_reaberto = json.loads(
    caminho_manifesto_mestre.read_text(encoding="utf-8")
)

validacoes_artefatos = {
    "contrato_existe": caminho_contrato.exists(),
    "particoes_existe": caminho_particoes.exists(),
    "metricas_existe": caminho_metricas.exists(),
    "predicoes_existe": caminho_predicoes.exists(),
    "manifesto_existe": caminho_manifesto.exists(),
    "manifesto_mestre_existe": caminho_manifesto_mestre.exists(),
    "metricas_mesmas_linhas": len(metricas_reabertas) == len(resultados_piloto),
    "predicoes_esperadas": len(predicoes_reabertas) == 2 * len(validacao),
    "status_coorte_manifesto_correto": (
        manifesto_reaberto["status_coorte"] == STATUS_COORTE
    ),
    "blocos_mestre_reabertos": (
        set(manifesto_mestre_reaberto["blocos"])
        == set(MAPA_BLOCO_PARA_PARTICAO) | {"exploracao"}
    ),
    "teste_declarado_fechado": (
        manifesto_reaberto["teste_aberto"] is False
        and manifesto_reaberto["particao_mestra"]["test_ref_fechado"]
        and manifesto_reaberto["particao_mestra"]["test_pandemic_fechado"]
    ),
}

checklist_final = {
    "contrato_validado": bool(contrato["resultado"].all()),
    "particoes_validas": sobreposicao_grupos == 0,
    "teste_nao_usado_em_modelagem": "teste" not in globals(),
    "dummy_executado": "metricas_dummy" in globals(),
    "logistica_executada": "metricas_logistica" in globals(),
    "probabilidades_validas": bool(validacoes_pos_treino["prob_logistica_validas"]),
    "artefatos_reabertos": all(validacoes_artefatos.values()),
    "resultados_com_status_correto": bool(
        (resultados_piloto["status_coorte"] == STATUS_COORTE).all()
    ),
}

checklist_final_df = pd.DataFrame({
    "item": checklist_final.keys(),
    "concluido": checklist_final.values(),
})
display(checklist_final_df)

if not all(checklist_final.values()):
    pendencias = [k for k, v in checklist_final.items() if not v]
    raise RuntimeError(f"Notebook 03 terminou com pendências: {pendencias}")

print("Notebook 03 executado com sucesso até Dummy + Regressão Logística piloto.")
print("O teste permaneceu fechado.")

In [ ]:
import inspect

import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

from src.evaluation.metrics import metricas_completas
from src.models.baseline import build_baseline_pipeline
from src.models.boosting import build_boosting_pipeline
from src.models.logistic import build_logistic_pipeline
from src.models.pipeline import build_preprocessor

OBJETOS_OBRIGATORIOS_COMPARACAO = [
    "X_treino",
    "y_treino",
    "X_validacao",
    "y_validacao",
    "treino",
    "validacao",
    "FEATURES_NUMERICAS",
    "FEATURES_CATEGORICAS",
    "FEATURES_PILOTO",
    "ALVO_MODELO",
    "COLUNA_ANO",
    "SEED",
    "STATUS_COORTE",
    "TABLES_DIR",
    "PREDICTIONS_DIR",
]

objetos_ausentes_comparacao = [
    nome
    for nome in OBJETOS_OBRIGATORIOS_COMPARACAO
    if nome not in globals()
]

if objetos_ausentes_comparacao:
    raise NameError(
        "O ambiente ainda não contém todos os objetos necessários "
        f"para a comparação dos pilotos: {objetos_ausentes_comparacao}"
    )

assinaturas_apis = {
    "build_preprocessor": inspect.signature(build_preprocessor),
    "build_baseline_pipeline": inspect.signature(build_baseline_pipeline),
    "build_logistic_pipeline": inspect.signature(build_logistic_pipeline),
    "build_boosting_pipeline": inspect.signature(build_boosting_pipeline),
    "metricas_completas": inspect.signature(metricas_completas),
    "OneHotEncoder": inspect.signature(OneHotEncoder),
    "OrdinalEncoder": inspect.signature(OrdinalEncoder),
    "HistGradientBoostingClassifier": inspect.signature(
        HistGradientBoostingClassifier
    ),
}

parametros_onehot = assinaturas_apis["OneHotEncoder"].parameters
parametros_ordinal = assinaturas_apis["OrdinalEncoder"].parameters
parametros_hgb = assinaturas_apis[
    "HistGradientBoostingClassifier"
].parameters

if "sparse_output" in parametros_onehot:
    PARAMETRO_ONEHOT_ESPARSIDADE = "sparse_output"
elif "sparse" in parametros_onehot:
    PARAMETRO_ONEHOT_ESPARSIDADE = "sparse"
else:
    raise TypeError(
        "A versão instalada do OneHotEncoder não possui parâmetro "
        "reconhecido para controlar a saída esparsa."
    )

SUPORTA_ORDINAL_ENCODED_MISSING = (
    "encoded_missing_value" in parametros_ordinal
)
SUPORTA_HGB_CLASS_WEIGHT = "class_weight" in parametros_hgb
SUPORTA_HGB_CATEGORICAL_FEATURES = (
    "categorical_features" in parametros_hgb
)

if not SUPORTA_HGB_CATEGORICAL_FEATURES:
    raise TypeError(
        "A versão instalada do HistGradientBoostingClassifier não aceita "
        "categorical_features. O piloto de boosting não deve ser executado "
        "com a construção atual."
    )

if set(FEATURES_NUMERICAS).intersection(FEATURES_CATEGORICAS):
    sobreposicao_features = sorted(
        set(FEATURES_NUMERICAS).intersection(FEATURES_CATEGORICAS)
    )
    raise RuntimeError(
        "Existem features classificadas simultaneamente como numéricas "
        f"e categóricas: {sobreposicao_features}"
    )

features_contrato_comparacao = (
    list(FEATURES_NUMERICAS)
    + list(FEATURES_CATEGORICAS)
)

if features_contrato_comparacao != list(FEATURES_PILOTO):
    raise RuntimeError(
        "A ordem de FEATURES_NUMERICAS + FEATURES_CATEGORICAS difere "
        "da ordem de FEATURES_PILOTO. A comparação não deve prosseguir "
        "com contratos divergentes."
    )

colunas_ausentes_treino = sorted(
    set(FEATURES_PILOTO).difference(X_treino.columns)
)
colunas_ausentes_validacao = sorted(
    set(FEATURES_PILOTO).difference(X_validacao.columns)
)

if colunas_ausentes_treino or colunas_ausentes_validacao:
    raise KeyError(
        "Existem features ausentes nas matrizes de modelagem. "
        f"Treino: {colunas_ausentes_treino}. "
        f"Validação: {colunas_ausentes_validacao}."
    )

if len(X_treino) != len(y_treino):
    raise RuntimeError(
        "X_treino e y_treino possuem quantidades de linhas diferentes."
    )

if len(X_validacao) != len(y_validacao):
    raise RuntimeError(
        "X_validacao e y_validacao possuem quantidades de linhas diferentes."
    )

if not X_treino.index.equals(y_treino.index):
    raise RuntimeError(
        "Os índices de X_treino e y_treino não estão alinhados."
    )

if not X_validacao.index.equals(y_validacao.index):
    raise RuntimeError(
        "Os índices de X_validacao e y_validacao não estão alinhados."
    )

classes_treino = sorted(
    pd.Series(y_treino)
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)
classes_validacao = sorted(
    pd.Series(y_validacao)
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

if classes_treino != [0, 1]:
    raise RuntimeError(
        "O treino precisa conter as duas classes binárias. "
        f"Classes observadas: {classes_treino}"
    )

if classes_validacao != [0, 1]:
    raise RuntimeError(
        "A validação precisa conter as duas classes binárias. "
        f"Classes observadas: {classes_validacao}"
    )

compatibilidade_ambiente = pd.DataFrame(
    [
        {
            "verificacao": "versao_sklearn",
            "resultado": sklearn.__version__,
        },
        {
            "verificacao": "parametro_onehot_esparsidade",
            "resultado": PARAMETRO_ONEHOT_ESPARSIDADE,
        },
        {
            "verificacao": "ordinal_encoded_missing_value",
            "resultado": SUPORTA_ORDINAL_ENCODED_MISSING,
        },
        {
            "verificacao": "hgb_class_weight",
            "resultado": SUPORTA_HGB_CLASS_WEIGHT,
        },
        {
            "verificacao": "hgb_categorical_features",
            "resultado": SUPORTA_HGB_CATEGORICAL_FEATURES,
        },
        {
            "verificacao": "objetos_obrigatorios_presentes",
            "resultado": not objetos_ausentes_comparacao,
        },
        {
            "verificacao": "contrato_features_consistente",
            "resultado": (
                features_contrato_comparacao
                == list(FEATURES_PILOTO)
            ),
        },
        {
            "verificacao": "indices_treino_alinhados",
            "resultado": X_treino.index.equals(y_treino.index),
        },
        {
            "verificacao": "indices_validacao_alinhados",
            "resultado": X_validacao.index.equals(
                y_validacao.index
            ),
        },
        {
            "verificacao": "teste_nao_materializado",
            "resultado": "teste" not in globals(),
        },
    ]
)

display(compatibilidade_ambiente)

print("Assinaturas das APIs centrais:")
for nome_api, assinatura_api in assinaturas_apis.items():
    print(f"- {nome_api}{assinatura_api}")

print("Classes no treino:", classes_treino)
print("Classes na validação:", classes_validacao)
print("Gate de compatibilidade concluído.")
print("O conjunto de teste não foi materializado.")

In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline

from src.evaluation.metrics import metricas_completas
from src.models.pipeline import build_preprocessor
from typing import cast

OBJETOS_PILOTOS_NECESSARIOS = [
    "X_treino",
    "y_treino",
    "X_validacao",
    "y_validacao",
    "treino",
    "validacao",
    "FEATURES_NUMERICAS",
    "FEATURES_CATEGORICAS",
    "FEATURES_PILOTO",
    "ALVO_MODELO",
    "COLUNA_ANO",
    "SEED",
    "STATUS_COORTE",
    "TABLES_DIR",
    "PREDICTIONS_DIR",
]

objetos_pilotos_ausentes = [
    nome
    for nome in OBJETOS_PILOTOS_NECESSARIOS
    if nome not in globals()
]

if objetos_pilotos_ausentes:
    raise NameError(
        "Execute o notebook até a conclusão de desenvolvimento antes da comparação. "
        f"Objetos ausentes: {objetos_pilotos_ausentes}"
    )

MAX_TREINO_PILOTO = 300_000
MAX_VALIDACAO_PILOTO = 120_000
VERSAO_COMPARACAO = "piloto_v2_cinco_familias"


def selecionar_amostra_estratificada(
    X: pd.DataFrame,
    y: pd.Series,
    anos: pd.Series,
    *,
    maximo: int,
    seed: int,
) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    if not (len(X) == len(y) == len(anos)):
        raise ValueError(
            "X, y e anos precisam possuir o mesmo número de linhas."
        )

    if maximo <= 0:
        raise ValueError(
            "O tamanho máximo da amostra precisa ser positivo."
        )

    y_alinhado = pd.Series(
        np.asarray(y, dtype=np.int8),
        index=X.index,
        name="alvo",
    )
    anos_alinhados = pd.Series(
        np.asarray(anos, dtype=np.int16),
        index=X.index,
        name="ano",
    )

    if len(X) <= maximo:
        return (
            X.copy(),
            y_alinhado.copy(),
            anos_alinhados.copy(),
        )

    estratos = (
        anos_alinhados.astype("string")
        + "__y"
        + y_alinhado.astype("string")
    )
    tabela_indices = pd.DataFrame(
        {
            "indice": X.index,
            "estrato": estratos.to_numpy(),
        },
        index=X.index,
    )

    contagens = tabela_indices["estrato"].value_counts()
    proporcoes = contagens / contagens.sum()
    cotas = np.floor(proporcoes * maximo).astype(int)
    cotas[cotas < 1] = 1
    diferenca = int(maximo - cotas.sum())

    if diferenca > 0:
        ordem = (
            (proporcoes * maximo - np.floor(proporcoes * maximo))
            .sort_values(ascending=False)
            .index
            .tolist()
        )
        for estrato in ordem[:diferenca]:
            cotas = pd.Series(
                np.floor(proporcoes.to_numpy() * maximo).astype(int),
                index=proporcoes.index,
                dtype=int,
            )
            cotas[cotas < 1] = 1
            diferenca = int(maximo - cotas.sum())
    elif diferenca < 0:
        ordem = (
            pd.Series(np.asarray(cotas), index=proporcoes.index)
            .sort_values(ascending=False)
            .index
            .tolist()
        )
        restante = -diferenca
        for estrato in ordem:
            if restante == 0:
                break
            cotas = pd.Series(
                cotas,
                index=proporcoes.index,
                dtype=int,
            )
            reducao = min(
                restante,
                max(int(cotas.loc[estrato]) - 1, 0),
            )
            cotas.loc[estrato] -= reducao
            restante -= reducao

    rng = np.random.default_rng(seed)
    indices_selecionados = []

    for estrato, bloco in tabela_indices.groupby(
        "estrato",
        sort=True,
        observed=True,
    ):
        candidatos = bloco["indice"].to_numpy()
        cotas_por_estrato = pd.Series(
            cotas,
            index=proporcoes.index,
        )
        quantidade = min(
            int(cotas_por_estrato.get(estrato, 0)),
            len(candidatos),
        )
        escolhidos = rng.choice(
            candidatos,
            size=quantidade,
            replace=False,
        )
        indices_selecionados.append(escolhidos)

    indices_finais = np.concatenate(indices_selecionados)
    rng.shuffle(indices_finais)

    return (
        X.loc[indices_finais].copy(),
        y_alinhado.loc[indices_finais].copy(),
        anos_alinhados.loc[indices_finais].copy(),
    )


anos_treino = treino.loc[X_treino.index, COLUNA_ANO]
anos_validacao = validacao.loc[X_validacao.index, COLUNA_ANO]

(
    X_treino_piloto,
    y_treino_piloto,
    anos_treino_piloto,
) = selecionar_amostra_estratificada(
    X_treino,
    y_treino,
    anos_treino,
    maximo=MAX_TREINO_PILOTO,
    seed=SEED,
)

(
    X_validacao_piloto,
    y_validacao_piloto,
    anos_validacao_piloto,
) = selecionar_amostra_estratificada(
    X_validacao,
    y_validacao,
    anos_validacao,
    maximo=MAX_VALIDACAO_PILOTO,
    seed=SEED + 1,
)

preprocessador_esparso = build_preprocessor(
    numericas=list(FEATURES_NUMERICAS),
    categoricas=list(FEATURES_CATEGORICAS),
    saida="esparsa",
)


def calcular_metricas_comparaveis(
    y_true,
    y_prob,
    *,
    limiar: float = 0.5,
) -> dict[str, float]:
    metricas = metricas_completas(
        y_true,
        y_prob,
        limiar=limiar,
    )

    if "auc" not in metricas:
        raise KeyError(
            "A função central de métricas não retornou 'auc'. "
            f"Chaves disponíveis: {sorted(metricas)}"
        )

    metricas["roc_auc"] = metricas["auc"]
    metricas["limiar_diagnostico"] = float(limiar)
    metricas["n_validacao"] = int(len(y_true))
    return metricas


def executar_modelo_comparavel(
    nome: str,
    pipeline: Pipeline,
) -> tuple[Pipeline, dict, pd.DataFrame]:
    inicio_fit = time.perf_counter()
    pipeline.fit(
        X_treino_piloto,
        y_treino_piloto,
    )
    duracao_fit = time.perf_counter() - inicio_fit

    inicio_pred = time.perf_counter()
    probabilidades = pipeline.predict_proba(
        X_validacao_piloto
    )[:, 1]
    duracao_pred = time.perf_counter() - inicio_pred

    if not np.isfinite(probabilidades).all():
        raise RuntimeError(
            f"{nome}: probabilidades não finitas."
        )

    if not ((probabilidades >= 0) & (probabilidades <= 1)).all():
        raise RuntimeError(
            f"{nome}: probabilidades fora de [0, 1]."
        )

    metricas = calcular_metricas_comparaveis(
        y_validacao_piloto,
        probabilidades,
        limiar=0.5,
    )
    metricas_saida = cast(dict[str, object], metricas)
    metricas_saida.update(
        {
            "modelo": nome,
            "status_coorte": STATUS_COORTE,
            "versao_execucao": VERSAO_COMPARACAO,
            "n_treino": int(len(X_treino_piloto)),
            "duracao_fit_s": float(duracao_fit),
            "duracao_pred_s": float(duracao_pred),
        }
    )

    predicoes = pd.DataFrame(
        {
            "id_registro": validacao.loc[X_validacao_piloto.index, "id_registro"].to_numpy(),
            "ano_internacao": anos_validacao_piloto.to_numpy(),
            "y_true": y_validacao_piloto.to_numpy(),
            "y_prob": probabilidades,
            "modelo": nome,
            "status_coorte": STATUS_COORTE,
            "versao_execucao": VERSAO_COMPARACAO,
        }
    )

    return pipeline, metricas, predicoes


resumo_amostras_comparacao = pd.DataFrame(
    {
        "particao": ["treino", "validacao"],
        "linhas_originais": [len(X_treino), len(X_validacao)],
        "linhas_piloto": [
            len(X_treino_piloto),
            len(X_validacao_piloto),
        ],
        "prevalencia_original": [
            float(y_treino.mean()),
            float(y_validacao.mean()),
        ],
        "prevalencia_piloto": [
            float(y_treino_piloto.mean()),
            float(y_validacao_piloto.mean()),
        ],
    }
)

display(resumo_amostras_comparacao)

if not np.allclose(
    resumo_amostras_comparacao["prevalencia_original"],
    resumo_amostras_comparacao["prevalencia_piloto"],
    atol=0.002,
):
    raise RuntimeError(
        "A amostra piloto alterou materialmente a prevalência do alvo."
    )

print("Amostras comparáveis e funções carregadas com sucesso.")


In [ ]:
# ============================================================
# 22. Executar cinco famílias na mesma amostra e partição
# ============================================================

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

from src.models.baseline import build_baseline_pipeline
from src.models.boosting import build_boosting_pipeline
from src.models.logistic import build_logistic_pipeline
from typing import Literal, cast

log_cfg_comparacao = CFG.get("models", {}).get("logistic", {})
boost_cfg_comparacao = CFG.get("models", {}).get("boosting", {})

solver_logistico = cast(
    Literal[
        "lbfgs",
        "liblinear",
        "newton-cg",
        "newton-cholesky",
        "sag",
        "saga",
    ],
    log_cfg_comparacao.get("solver", "lbfgs"),
)

modelos_piloto = {
    "dummy_prior_v2": build_baseline_pipeline(
        strategy="prior",
        preprocessor=clone(preprocessador_esparso),
    ),
    "logistica_C1_v2": build_logistic_pipeline(
        C=1.0,
        penalty=log_cfg_comparacao.get("penalty"),
        solver=solver_logistico,
        max_iter=500,
        class_weight="balanced",
        preprocessor=clone(preprocessador_esparso),
    ),
    "arvore_rasa_v2": Pipeline(
        [
            ("pre", clone(preprocessador_esparso)),
            (
                "clf",
                DecisionTreeClassifier(
                    max_depth=6,
                    min_samples_leaf=500,
                    class_weight="balanced",
                    random_state=SEED,
                ),
            ),
        ]
    ),
    "random_forest_contida_v2": Pipeline(
        [
            ("pre", clone(preprocessador_esparso)),
            (
                "clf",
                RandomForestClassifier(
                    n_estimators=120,
                    max_depth=12,
                    min_samples_leaf=100,
                    max_features="sqrt",
                    max_samples=0.70,
                    class_weight="balanced_subsample",
                    n_jobs=-1,
                    random_state=SEED,
                ),
            ),
        ]
    ),
    "hist_gradient_boosting_v2": build_boosting_pipeline(
        learning_rate=float(
            boost_cfg_comparacao.get("learning_rate") or 0.08
        ),
        max_iter=int(boost_cfg_comparacao.get("max_iter") or 150),
        max_depth=boost_cfg_comparacao.get("max_depth"),
        max_leaf_nodes=int(
            boost_cfg_comparacao.get("max_leaf_nodes") or 31
        ),
        min_samples_leaf=int(
            boost_cfg_comparacao.get("min_samples_leaf") or 100
        ),
        l2_regularization=float(
            boost_cfg_comparacao.get("l2_regularization") or 0.10
        ),
        early_stopping=bool(
            boost_cfg_comparacao.get("early_stopping", True)
        ),
        validation_fraction=float(
            boost_cfg_comparacao.get("validation_fraction") or 0.10
        ),
        n_iter_no_change=int(
            boost_cfg_comparacao.get("n_iter_no_change") or 20
        ),
        class_weight=boost_cfg_comparacao.get("class_weight", "balanced"),
        preprocessor=None,
    ),
}

pipelines_cinco_familias = {}
metricas_cinco_familias = []
predicoes_cinco_familias = []

for nome_modelo, pipeline_modelo in modelos_piloto.items():
    print("Executando:", nome_modelo)
    (
        pipeline_ajustado,
        metricas_modelo,
        predicoes_modelo,
    ) = executar_modelo_comparavel(
        nome_modelo,
        pipeline_modelo,
    )

    pipelines_cinco_familias[nome_modelo] = pipeline_ajustado
    metricas_cinco_familias.append(metricas_modelo)
    predicoes_cinco_familias.append(predicoes_modelo)

    print(
        "PR-AUC:",
        round(metricas_modelo["pr_auc"], 6),
        "| ROC-AUC:",
        round(metricas_modelo["roc_auc"], 6),
        "| Brier:",
        round(metricas_modelo["brier"], 6),
        "| fit (s):",
        round(metricas_modelo["duracao_fit_s"], 2),
    )

resultados_cinco_familias = (
    pd.DataFrame(metricas_cinco_familias)
    .sort_values(
        ["pr_auc", "brier"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

predicoes_cinco_familias_df = pd.concat(
    predicoes_cinco_familias,
    ignore_index=True,
)

colunas_resultado = [
    "modelo",
    "pr_auc",
    "roc_auc",
    "brier",
    "recall",
    "precision",
    "f1",
    "accuracy",
    "n_treino",
    "n_validacao",
    "duracao_fit_s",
    "duracao_pred_s",
    "status_coorte",
    "versao_execucao",
]

colunas_resultado_ausentes = [
    coluna
    for coluna in colunas_resultado
    if coluna not in resultados_cinco_familias.columns
]

if colunas_resultado_ausentes:
    raise KeyError(
        "A comparação não produziu todas as colunas esperadas: "
        f"{colunas_resultado_ausentes}"
    )

display(resultados_cinco_familias[colunas_resultado])

if len(resultados_cinco_familias) != 5:
    raise RuntimeError(
        "A tabela comparativa não contém as cinco famílias esperadas."
    )

if predicoes_cinco_familias_df["modelo"].nunique() != 5:
    raise RuntimeError(
        "As predições não contêm as cinco famílias esperadas."
    )

if not predicoes_cinco_familias_df["y_prob"].between(0, 1).all():
    raise RuntimeError(
        "Existem probabilidades inválidas na comparação."
    )

print("As cinco famílias foram executadas na mesma amostra comparável.")



In [ ]:
# ============================================================
# 18. Diagnósticos de discriminação, calibração e estabilidade anual
# ============================================================

import matplotlib.pyplot as plt

from sklearn.calibration import calibration_curve
from sklearn.metrics import precision_recall_curve, roc_curve


FIGURES_DIR = ROOT_DIR / "reports" / "figures" / "supervisionado_03"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

modelos_avaliaveis = (
    resultados_cinco_familias.loc[
        resultados_cinco_familias["modelo"].ne("dummy_prior_v2"),
        "modelo",
    ]
    .tolist()
)

figura_curvas, eixos_curvas = plt.subplots(
    1,
    3,
    figsize=(18, 5.5),
)

prevalencia_validacao_piloto = float(
    y_validacao_piloto.mean()
)

for modelo in modelos_avaliaveis:
    dados_modelo = predicoes_cinco_familias_df.loc[
        predicoes_cinco_familias_df["modelo"].eq(modelo)
    ]

    precisao, recall, _ = precision_recall_curve(
        dados_modelo["y_true"],
        dados_modelo["y_prob"],
    )

    falso_positivo, verdadeiro_positivo, _ = roc_curve(
        dados_modelo["y_true"],
        dados_modelo["y_prob"],
    )

    fracao_positivos, media_prevista = calibration_curve(
        dados_modelo["y_true"],
        dados_modelo["y_prob"],
        n_bins=10,
        strategy="quantile",
    )

    eixos_curvas[0].plot(
        recall,
        precisao,
        label=modelo,
    )

    eixos_curvas[1].plot(
        falso_positivo,
        verdadeiro_positivo,
        label=modelo,
    )

    eixos_curvas[2].plot(
        media_prevista,
        fracao_positivos,
        marker="o",
        label=modelo,
    )


eixos_curvas[0].axhline(
    prevalencia_validacao_piloto,
    linestyle="--",
    color="gray",
    linewidth=1,
    label="prevalência",
)
eixos_curvas[0].set_title("Curvas Precisão-Recall")
eixos_curvas[0].set_xlabel("Recall")
eixos_curvas[0].set_ylabel("Precisão")

eixos_curvas[1].plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="gray",
    linewidth=1,
)
eixos_curvas[1].set_title("Curvas ROC")
eixos_curvas[1].set_xlabel("Taxa de falsos positivos")
eixos_curvas[1].set_ylabel("Taxa de verdadeiros positivos")

eixos_curvas[2].plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="black",
    linewidth=1,
    label="calibração perfeita",
)
eixos_curvas[2].set_title("Curvas de calibração")
eixos_curvas[2].set_xlabel("Probabilidade média prevista")
eixos_curvas[2].set_ylabel("Fração observada de positivos")

for eixo in eixos_curvas:
    eixo.grid(alpha=0.2)
    eixo.legend(fontsize=8)

figura_curvas.suptitle(
    "Comparação de desenvolvimento das famílias na mesma validação"
)
figura_curvas.tight_layout()

caminho_figura_curvas = (
    FIGURES_DIR
    / "03_pilotos_curvas_pr_roc_calibracao.png"
)
figura_curvas.savefig(
    caminho_figura_curvas,
    dpi=160,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()
plt.close(figura_curvas)


metricas_por_ano = []

pares_modelo_ano = (
    predicoes_cinco_familias_df[
        ["modelo", "ano_internacao"]
    ]
    .dropna()
    .drop_duplicates()
    .to_dict(orient="records")
)

for par in pares_modelo_ano:
    modelo = par["modelo"]
    ano = par["ano_internacao"]

    grupo = predicoes_cinco_familias_df.loc[
        predicoes_cinco_familias_df["modelo"].eq(modelo)
        & predicoes_cinco_familias_df["ano_internacao"].eq(ano)
    ]

    if grupo.empty:
        continue

    y_ano = grupo["y_true"].to_numpy(dtype=np.int8)
    prob_ano = grupo["y_prob"].to_numpy(dtype=float)

    if np.unique(y_ano).size < 2:
        roc_auc_ano = np.nan
        pr_auc_ano = np.nan
    else:
        roc_auc_ano = float(roc_auc_score(y_ano, prob_ano))
        pr_auc_ano = float(average_precision_score(y_ano, prob_ano))

    metricas_por_ano.append(
        {
            "modelo": modelo,
            "ano_internacao": int(ano),
            "n": int(len(grupo)),
            "positivos": int(y_ano.sum()),
            "prevalencia": float(y_ano.mean()),
            "pr_auc": pr_auc_ano,
            "roc_auc": roc_auc_ano,
            "brier": float(brier_score_loss(y_ano, prob_ano)),
            "probabilidade_media": float(prob_ano.mean()),
        }
    )

metricas_por_ano_df = pd.DataFrame(
    metricas_por_ano
)

variacao_temporal_modelos = (
    metricas_por_ano_df.loc[
        metricas_por_ano_df["modelo"].ne("dummy_prior_v2")
    ]
    .groupby("modelo", observed=True)
    .agg(
        pr_auc_min=("pr_auc", "min"),
        pr_auc_max=("pr_auc", "max"),
        pr_auc_media=("pr_auc", "mean"),
        roc_auc_min=("roc_auc", "min"),
        roc_auc_max=("roc_auc", "max"),
        brier_min=("brier", "min"),
        brier_max=("brier", "max"),
    )
    .reset_index()
)

variacao_temporal_modelos["amplitude_pr_auc"] = (
    variacao_temporal_modelos["pr_auc_max"]
    - variacao_temporal_modelos["pr_auc_min"]
)

variacao_temporal_modelos["amplitude_roc_auc"] = (
    variacao_temporal_modelos["roc_auc_max"]
    - variacao_temporal_modelos["roc_auc_min"]
)

figura_temporal, eixos_temporais = plt.subplots(
    1,
    2,
    figsize=(14, 5),
)

for modelo in modelos_avaliaveis:
    dados_modelo = metricas_por_ano_df.loc[
        metricas_por_ano_df["modelo"].eq(modelo)
    ].sort_values("ano_internacao")

    eixos_temporais[0].plot(
        dados_modelo["ano_internacao"],
        dados_modelo["pr_auc"],
        marker="o",
        label=modelo,
    )

    eixos_temporais[1].plot(
        dados_modelo["ano_internacao"],
        dados_modelo["brier"],
        marker="o",
        label=modelo,
    )


eixos_temporais[0].set_title("PR-AUC por ano")
eixos_temporais[0].set_xlabel("Ano de internação")
eixos_temporais[0].set_ylabel("PR-AUC")

eixos_temporais[1].set_title("Brier por ano")
eixos_temporais[1].set_xlabel("Ano de internação")
eixos_temporais[1].set_ylabel("Brier")

for eixo in eixos_temporais:
    eixo.grid(alpha=0.2)
    eixo.legend(fontsize=8)

figura_temporal.suptitle(
    "Estabilidade temporal de desenvolvimento na validação"
)
figura_temporal.tight_layout()

caminho_figura_temporal = (
    FIGURES_DIR
    / "03_pilotos_metricas_por_ano.png"
)
figura_temporal.savefig(
    caminho_figura_temporal,
    dpi=160,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()
plt.close(figura_temporal)


display(
    metricas_por_ano_df.sort_values(
        ["modelo", "ano_internacao"]
    ).reset_index(drop=True)
)

display(
    variacao_temporal_modelos.sort_values(
        ["amplitude_pr_auc", "pr_auc_media"],
        ascending=[True, False],
    ).reset_index(drop=True)
)

print("Figuras gravadas:")
print("-", caminho_figura_curvas)
print("-", caminho_figura_temporal)


In [ ]:
# ============================================================
# 19. Persistir a comparação de desenvolvimento
# ============================================================

import json


timestamp_comparacao = pd.Timestamp.now(
    tz="UTC"
).strftime("%Y%m%dT%H%M%SZ")

prefixo_comparacao = (
    f"03_cinco_familias_{STATUS_COORTE}_"
    f"{timestamp_comparacao}"
)

caminho_resultados_cinco = (
    Path(TABLES_DIR)
    / f"{prefixo_comparacao}_metricas.csv"
)

caminho_metricas_ano = (
    Path(TABLES_DIR)
    / f"{prefixo_comparacao}_metricas_por_ano.csv"
)

caminho_variacao_temporal = (
    Path(TABLES_DIR)
    / f"{prefixo_comparacao}_variacao_temporal.csv"
)

caminho_amostras = (
    Path(TABLES_DIR)
    / f"{prefixo_comparacao}_amostras.csv"
)

caminho_predicoes_cinco = (
    Path(PREDICTIONS_DIR)
    / f"{prefixo_comparacao}_predicoes_validacao.parquet"
)

caminho_manifesto_cinco = (
    Path(TABLES_DIR)
    / f"{prefixo_comparacao}_manifesto.json"
)

resultados_cinco_familias.to_csv(
    caminho_resultados_cinco,
    index=False,
    encoding="utf-8-sig",
)

metricas_por_ano_df.to_csv(
    caminho_metricas_ano,
    index=False,
    encoding="utf-8-sig",
)

variacao_temporal_modelos.to_csv(
    caminho_variacao_temporal,
    index=False,
    encoding="utf-8-sig",
)

resumo_amostras_comparacao.to_csv(
    caminho_amostras,
    index=False,
    encoding="utf-8-sig",
)

predicoes_cinco_familias_df.to_parquet(
    caminho_predicoes_cinco,
    index=False,
)


ranking_desenvolvimento = (
    resultados_cinco_familias.loc[
        resultados_cinco_familias["modelo"].ne("dummy_prior_v2")
    ]
    .copy()
    .sort_values(
        ["pr_auc", "brier", "duracao_fit_s"],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)

shortlist_desenvolvimento = ranking_desenvolvimento.head(2)[
    "modelo"
].tolist()

manifesto_cinco = {
    "status_coorte": STATUS_COORTE,
    "versao_execucao": VERSAO_COMPARACAO,
    "carater": (
        "comparacao_definitiva_de_integracao"
        if TIPO_COORTE == "final"
        else "comparacao_de_desenvolvimento"
    ),
    "seed": int(SEED),
    "n_treino_piloto": int(len(X_treino_piloto)),
    "n_validacao_piloto": int(len(X_validacao_piloto)),
    "features": list(FEATURES_PILOTO),
    "modelos": list(modelos_piloto.keys()),
    "shortlist_desenvolvimento": shortlist_desenvolvimento,
    "teste_aberto": False,
    "limiar_selecionado": False,
    "calibracao_aplicada": False,
    "tuning_realizado": False,
    "observacoes": [
        (
            "Resultados produzidos com a coorte definitiva publicada."
            if TIPO_COORTE == "final"
            else "Resultados dependem da coorte de desenvolvimento."
        ),
        "Amostras comparáveis foram usadas para controlar custo.",
        "A shortlist não congela finalistas.",
        "O teste permaneceu fechado.",
        "Calibração e limiar pertencem à próxima etapa.",
    ],
    "artefatos": {
        "metricas": str(caminho_resultados_cinco),
        "metricas_por_ano": str(caminho_metricas_ano),
        "variacao_temporal": str(caminho_variacao_temporal),
        "amostras": str(caminho_amostras),
        "predicoes_validacao": str(caminho_predicoes_cinco),
        "figura_curvas": str(caminho_figura_curvas),
        "figura_temporal": str(caminho_figura_temporal),
    },
}

caminho_manifesto_cinco.write_text(
    json.dumps(
        manifesto_cinco,
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
    encoding="utf-8",
)


# Round-trip dos principais artefatos.
resultados_reabertos = pd.read_csv(
    caminho_resultados_cinco,
    encoding="utf-8-sig",
)

predicoes_reabertas = pd.read_parquet(
    caminho_predicoes_cinco
)

manifesto_reaberto_cinco = json.loads(
    caminho_manifesto_cinco.read_text(
        encoding="utf-8"
    )
)

validacoes_comparacao = {
    "cinco_modelos_registrados": len(resultados_reabertos) == 5,
    "cinco_modelos_com_predicoes": predicoes_reabertas["modelo"].nunique() == 5,
    "probabilidades_validas": predicoes_reabertas["y_prob"].between(0, 1).all(),
    "teste_permaneceu_fechado": manifesto_reaberto_cinco["teste_aberto"] is False,
    "status_coorte_confere": (
        manifesto_reaberto_cinco["status_coorte"]
        == STATUS_COORTE
    ),
    "sem_tuning": manifesto_reaberto_cinco["tuning_realizado"] is False,
    "sem_limiar_final": manifesto_reaberto_cinco["limiar_selecionado"] is False,
    "figuras_existentes": caminho_figura_curvas.exists() and caminho_figura_temporal.exists(),
}

validacoes_comparacao_df = pd.DataFrame(
    {
        "verificacao": validacoes_comparacao.keys(),
        "resultado": validacoes_comparacao.values(),
    }
)

display(validacoes_comparacao_df)

if not all(validacoes_comparacao.values()):
    falhas = [
        chave
        for chave, valor in validacoes_comparacao.items()
        if not valor
    ]
    raise RuntimeError(
        "Falhas na comparação da coorte: "
        f"{falhas}"
    )

print("Shortlist de desenvolvimento, sem congelamento:", shortlist_desenvolvimento)
print("Artefatos da comparação gravados com sucesso.")
print("O teste permaneceu fechado.")
print(
    "Próximo gate: confirmar os finalistas na coorte definitiva "
    "antes do tuning limitado, da calibração e do limiar."
)


## Conclusão da comparação de desenvolvimento

A execução usa o artefato supervisionado definitivo e mantém a separação
entre preparação dos dados, treinamento, validação e acesso ao holdout.

### Evidências produzidas

- o artefato central foi carregado com `cohort_kind="final"`;
- `train_ref` foi usado para ajuste;
- `val_tune`, `val_cal` e `val_meas` permaneceram identificados;
- as previsões preservam `id_registro` e o bloco de origem;
- os modelos foram comparados com a mesma representação e as mesmas amostras;
- os artefatos foram reabertos e verificados após a persistência;
- `test_ref` e `test_pandemic` permaneceram fechados.

### Limites de interpretação

Os resultados desta etapa são resultados de desenvolvimento. Eles não são
métricas finais de teste, não definem calibração ou limiar operacional e não
devem ser tratados como validação clínica. Diferenças por ano ou subgrupo são
diagnósticos de estabilidade, não evidência causal.

### Continuidade

A próxima decisão deve congelar a shortlist e os hiperparâmetros usando apenas
os blocos autorizados. A abertura do teste continua condicionada ao gate
formal e deve ocorrer uma única vez.